# Introduction to Data Science 2025

# Week 6: Recap

## Exercise 1 | Linear regression with feature selection

Download the [TED Talks](https://www.kaggle.com/rounakbanik/ted-talks) dataset from Kaggle. Your task is to predict both the ratings and the number of views of a given TED talk. You should focus only on the <span style="font-weight: bold">ted_main</span> table.

1. Download the data, extract the following ratings from column <span style="font-weight: bold">ratings</span>: <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, <span style="font-weight: bold">Inspiring</span>. Store these values into respective columns so that they are easier to access. Next, extract the tags from column <span style="font-weight: bold">tags</span>. Count the number of occurrences of each tag and select the top-100 most common tags. Create a binary variable for each of these and include them in your data table, so that you can directly see whether a given tag (among the top-100 tags) is used in a given TED talk or not. The dataset you compose should have dimension (2550, 104), and comprise of the 'views' column, the three columns with counts of "Funny", "Confusing and "Inspiring" ratings, and 100 columns which one-hot encode the top-100 most common tag columns.


In [50]:
import json
import ast

import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

from PIL import Image
import numpy as np
from tpot import TPOTClassifier

In [51]:
df = pd.read_csv("ted_main.csv")

# Add ratings columns
mapping = df["ratings"].map(lambda x: json.loads(x.replace("'", '"')))
ratings = [{x["name"]: x["count"] for x in entry if x["name"] in ["Funny", "Confusing", "Inspiring"]} for entry in mapping]

temp = pd.concat([df, pd.json_normalize(ratings)], axis=1)

In [52]:
# Add tags
# Unfortunately this seems to require ast :(
tags_df = df["tags"].apply(ast.literal_eval).explode()
# TOp 100
top_100 = tags_df.value_counts().head(100).index
tags_out = pd.crosstab(tags_df.index, tags_df).reindex(columns=top_100, fill_value=0)

out = pd.concat([temp[["views", "Funny", "Confusing", "Inspiring"]].reset_index(drop=True), tags_out.reset_index(drop=True)], axis=1)

2. Construct a linear regression model to predict the number of views based on the data in the <span style="font-weight: bold">ted_main</span> table, including the binary variables for the top-100 tags that you just created.

In [53]:
# Unsure if we want to use the resulting dataframe for prediction
X = out.iloc[:, 1:]
y = out.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

model.predict(X_test)

array([ 1189960.74685409,   780378.36035072,   889906.512724  ,
         627896.79478416,   494960.48940309,  1071470.58259452,
        7035606.33622597,   897857.24023395,   -42341.6334436 ,
         815874.46602044,  3395315.74866413,  1073865.31332446,
         275913.97767785,   397885.77488917,  1091122.98496346,
         648793.97399282,   745523.33247083,  3031180.64567207,
        1008398.95698448,  1398640.08123908,  6163384.45187675,
        1449837.34854714,  1039123.3831929 ,  1291330.24846931,
         859651.06205817,   896723.87921139,   518422.23094087,
        4538844.58471603,  1001195.69190667,  1380139.54526885,
        1813615.78790979,  1325309.9278077 ,  1369949.32700917,
        2250129.23328146,   220673.97701367,   722669.0158811 ,
        1107798.32964923,  2778525.72643325,  1363175.31673503,
         833656.18600557,  1574811.79709815,  1433009.51497354,
         780155.3641049 ,   613388.23242984,  1252081.15768109,
        1224484.15026419,  1991601.69594

3. Do the same for the <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, and <span style="font-weight: bold">Inspiring</span> ratings.

In [54]:
X = out.iloc[:, 4:]
# All three at once
y = out.iloc[:, 1:4]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

model.predict(X_test)

array([[-129.57085941,    7.84005592,   66.02387441],
       [ -68.68465217,    9.01925231,  -38.71572658],
       [  -4.99858439,   17.89085912,  305.28935879],
       ...,
       [-157.45019414,   11.89088554,   88.19059749],
       [  49.63826357,   13.70565026,  451.46681399],
       [ 107.506043  ,   17.33998832,  397.6117595 ]], shape=(638, 3))

4. You will probably notice that most of the tags are not useful in predicting the views and the ratings. You should use some kind of variable selection to prune the set of tags that are included in the model. You can use for example classical p-values or more modern [LASSO](https://en.wikipedia.org/wiki/Lasso_(statistics)) techniques. Which tags are the best predictors of each of the response variables?

In [55]:
X = out.iloc[:, 4:]
y = out.iloc[:, 0]

# Avoids manual alpha tuning
lasso = LassoCV(random_state=1).fit(X, y)

pd.Series(lasso.coef_, index=X.columns).sort_values(ascending=False)

psychology       1.547282e+06
work             7.895974e+05
motivation       5.747320e+05
culture          5.642394e+05
brain            4.908179e+05
                     ...     
economics       -2.262338e+05
politics        -2.366574e+05
design          -2.661477e+05
art             -3.137049e+05
global issues   -4.017182e+05
Length: 100, dtype: float64

5. Produce summaries of your results. Could you recommend good tags – or tags to avoid! – for speakers targeting plenty of views and/or certain ratings?

In [56]:
"""
Top 5 tags seem to be psychology, work, motivation, culture and brain.
Similarly, the bottom 5 tags are global issues, art, design, politics and economics.

This applies when targeting views.
"""

'\nTop 5 tags seem to be psychology, work, motivation, culture and brain.\nSimilarly, the bottom 5 tags are global issues, art, design, politics and economics.\n\nThis applies when targeting views.\n'

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 2 | Symbol classification (part 2)

Note that it is strongly recommended to use Python in this exercise. However, if you can find a suitable AutoML implementation for your favorite language (e.g [here](http://h2o-release.s3.amazonaws.com/h2o/master/3888/docs-website/h2o-docs/automl.html) seems to be one for R) then you are free to use that language as well.

Use the preprocessed data from week 3 (you can also produce them using the example solutions of week 3).

1. This time train a *random forest classifier* on the data. A random forest is a collection of *decision trees*, which makes it an *ensemble* of classifiers. Each tree uses a random subset of the features to make its prediction. Without tuning any parameters, how is the accuracy?

In [57]:
# Sample solutions from week 3
df = pd.read_csv('hasy-data-labels.csv')
df = df[df.symbol_id >= 70]
df = df[df.symbol_id <= 79]
img_data = []
targets = []

# read the image data
for index, row in df.iterrows():
    img = Image.open(row['path'])
    img = img.convert("L")
    img = np.array(img.getdata())
    img_data.append(img)
    targets.append(row['latex'])

img_data = np.array(img_data)

training_data, test_data, train_target, test_target = train_test_split(img_data, targets, train_size=0.8)

model = LogisticRegression(max_iter = 500, multi_class='multinomial')
lr = model.fit(training_data, train_target)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(training_data, train_target)

predictedLr = lr.predict(test_data)
predictedDummy = dummy.predict(test_data)

print('Real data')
print(np.asarray(test_target)[:10], '\n')

print('Majority classifier result')
print(predictedDummy[:10], '\n')

print('Logistic regression predictor result')
print(predictedLr[:10], '\n')

print('Majority classifier accuracy:', accuracy_score(test_target, predictedDummy))
print('Logistic regression accuracy:', accuracy_score(test_target, predictedLr))

/Users/tuome/.local/share/mamba/envs/tpot/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Real data
['5' '3' '6' '9' '6' '0' '6' '9' '3' '2'] 

Majority classifier result
['0' '0' '0' '0' '0' '0' '0' '0' '0' '0'] 

Logistic regression predictor result
['5' '3' '2' '9' '6' '0' '5' '9' '9' '2'] 

Majority classifier accuracy: 0.12254901960784313
Logistic regression accuracy: 0.8235294117647058


In [58]:
rfc = RandomForestClassifier(random_state=1)
rfc.fit(training_data, train_target)
pred = rfc.predict(test_data)

print("RandomForestClassifier accuracy:", accuracy_score(test_target, pred))

RandomForestClassifier accuracy: 0.8186274509803921


2. The amount of trees to use as a part of the random forest is an example of a hyperparameter, because it is a parameter that is set prior to the learning process. In contrast, a parameter is a value in the model that is learned from the data. Train 20 classifiers, with varying amounts of decision trees starting from 10 up until 200, and plot the test accuracy as a function of the amount of classifiers. Does the accuracy keep increasing? Is more better?

In [59]:
tree_count = [x for x in range(10, 210, 10)]

for i, count in enumerate(tree_count):
    rfc = RandomForestClassifier(n_estimators=count, random_state=1)
    rfc.fit(training_data, train_target)
    pred = rfc.predict(test_data)
    print(f"RandomForestClassifier accuracy ({count}):", accuracy_score(test_target, pred))

RandomForestClassifier accuracy (10): 0.7598039215686274
RandomForestClassifier accuracy (20): 0.7794117647058824
RandomForestClassifier accuracy (30): 0.8333333333333334
RandomForestClassifier accuracy (40): 0.8186274509803921
RandomForestClassifier accuracy (50): 0.8333333333333334
RandomForestClassifier accuracy (60): 0.8333333333333334
RandomForestClassifier accuracy (70): 0.8235294117647058
RandomForestClassifier accuracy (80): 0.8333333333333334
RandomForestClassifier accuracy (90): 0.8284313725490197
RandomForestClassifier accuracy (100): 0.8186274509803921
RandomForestClassifier accuracy (110): 0.8333333333333334
RandomForestClassifier accuracy (120): 0.8382352941176471
RandomForestClassifier accuracy (130): 0.8333333333333334
RandomForestClassifier accuracy (140): 0.8382352941176471
RandomForestClassifier accuracy (150): 0.8480392156862745
RandomForestClassifier accuracy (160): 0.8333333333333334
RandomForestClassifier accuracy (170): 0.8431372549019608
RandomForestClassifier 

3. If we had picked the amount of decision trees by taking the value with the best test accuracy from the last plot, we would have *overfit* our hyperparameters to the test data. Can you see why it is a mistake to tune hyperparameters of your model by using the test data?

We can't tell how well a model is performing by optimizing for some specific test_data input, since we'll overfit the model.

4. Reshuffle and resplit the data so that it is divided in 3 parts: training (80%), validation (10%) and test (10%). Repeatedly train a model of your choosing (e.g random forest) on the training data, and evaluate it’s performance on the validation set, while tuning the hyperparameters so that the accuracy on the validation set increases. Then, finally evaluate the performance of your model on the test data. What can you say in terms of the generalization of your model?

In [60]:
training_data, split_data, train_target, split_target = train_test_split(img_data, targets, train_size=0.8)
validation_data, test_data, validation_target, test_target = train_test_split(split_data, split_target, train_size=0.5)

tree_count = [x for x in range(10, 210, 10)]

for i, count in enumerate(tree_count):
    rfc = RandomForestClassifier(n_estimators=count, random_state=1)
    rfc.fit(training_data, train_target)
    pred = rfc.predict(test_data)
    print(f"RandomForestClassifier accuracy ({count}):", accuracy_score(test_target, pred))

RandomForestClassifier accuracy (10): 0.7450980392156863
RandomForestClassifier accuracy (20): 0.8431372549019608
RandomForestClassifier accuracy (30): 0.8529411764705882
RandomForestClassifier accuracy (40): 0.8725490196078431
RandomForestClassifier accuracy (50): 0.9019607843137255
RandomForestClassifier accuracy (60): 0.9215686274509803
RandomForestClassifier accuracy (70): 0.9117647058823529
RandomForestClassifier accuracy (80): 0.8921568627450981
RandomForestClassifier accuracy (90): 0.8823529411764706
RandomForestClassifier accuracy (100): 0.8921568627450981
RandomForestClassifier accuracy (110): 0.9019607843137255
RandomForestClassifier accuracy (120): 0.9117647058823529
RandomForestClassifier accuracy (130): 0.9215686274509803
RandomForestClassifier accuracy (140): 0.9117647058823529
RandomForestClassifier accuracy (150): 0.9215686274509803
RandomForestClassifier accuracy (160): 0.9117647058823529
RandomForestClassifier accuracy (170): 0.8823529411764706
RandomForestClassifier 

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 3 | TPOT

The process of picking a suitable model, evaluating its performance and tuning the hyperparameters is very time consuming. A new idea in machine learning is the concept of automating this by using an optimization algorithm to find the best model in the space of models and their hyperparameters. Have a look at [TPOT](https://github.com/EpistasisLab/tpot), an automated ML solution that finds a good model and a good set of hyperparameters automatically. Try it on this data, it should outperform simple models like the ones we tried easily. Note that running the algorithm might take a while, depending on the strength of your computer. 

*Note*: In case it is running for too long, try checking if the parameters you are using when calling TPOT are reasonable, i.e. try reducing number of ‘generations’ or ‘population_size’. TPOT uses cross-validation internally, so we don’t need our own validation set.

In [61]:
# Without LabelEncoder, accuracy_score is always 0
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(targets)
training_data, test_data, train_target, test_target = train_test_split(img_data, y, train_size=0.8)

tpot = TPOTClassifier(generations=2, population_size=15)

tpot.fit(training_data, train_target)

pred = tpot.predict(test_data)
print("TPOTClassifier accuracy:", accuracy_score(test_target, pred))

/Users/tuome/.local/share/mamba/envs/tpot/lib/python3.13/site-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
/Users/tuome/.local/share/mamba/envs/tpot/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 51473 instead
  warnings.warn(
Generation:   0%|          | 0/3 [22:44:36<?, ?it/s]
/Users/tuome/.local/share/mamba/envs/tpot/lib/python3.13/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Generati

TPOTClassifier accuracy: 0.8872549019607843


**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**